In [ ]:
# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 2. 파일 업로드
from google.colab import files
import json

uploaded = files.upload()

In [ ]:
# 3. 필요한 패키지 설치
!pip install --upgrade transformers datasets accelerate --quiet

In [ ]:
# 4. 주요 라이브러리 로딩
import json
import torch
from datasets import Dataset
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast, Trainer, TrainingArguments, DataCollatorForLanguageModeling, EarlyStoppingCallback

In [ ]:
# 5. 모델 및 토크나이저 로딩
model_name = "skt/kogpt2-base-v2"
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.add_special_tokens({'eos_token': '</s>', 'pad_token': '<pad>', 'additional_special_tokens': ['<END>']})
model.resize_token_embeddings(len(tokenizer))

In [ ]:
# 6. Train Set 로딩
with open("augmented_logic.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

In [ ]:
# 7. Validation Set 로딩
with open("logic.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)

In [ ]:
# 8. 문제생성용 프롬프트
def format_data(data):
    return [{"Text": f"문제: {item['Question']}\n"} for item in data]

formatted_train_data = format_data(train_data)
formatted_val_data = format_data(val_data)

In [ ]:
# 9. 토크나이즈 함수
def tokenize(example):
    return tokenizer(example["Text"], padding="max_length", truncation=True, max_length=512)

In [ ]:
# 10. Dataset 변환 및 전처리
train_dataset = Dataset.from_list(formatted_train_data)
val_dataset = Dataset.from_list(formatted_val_data)

tokenized_train_dataset = train_dataset.map(tokenize, batched=False)
tokenized_val_dataset = val_dataset.map(tokenize, batched=False)

In [ ]:
# 11. Trainer 준비
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/KoGPT/CheckPoint/Logic/Problem",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=1,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
    fp16=torch.cuda.is_available(),
    overwrite_output_dir=True
)

In [ ]:
# 12. Trainer 구성
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

In [ ]:
# 13. 학습 실행
trainer.train()

In [ ]:
# 14. 모델 저장
model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Logic/Problem")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Logic/Problem")